# Sustainable Agriculture

## 📊 Business Context
Compare crop yields.

**Analytical Approach:** Hypothesis
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import TTestIndPower

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Data Generation
def generate_ab_data(n=1000):
    np.random.seed(42)
    # Control Group (Baseline)
    control = np.random.normal(100, 20, n)
    # Treatment Group (Intervention - slight uplift)
    treatment = np.random.normal(103, 22, n)
    
    # Create DataFrame
    df = pd.DataFrame({
        'Group': ['Control']*n + ['Treatment']*n,
        'Yield_Per_Hectare': np.concatenate([control, treatment])
    })
    return df

df = generate_ab_data(200)
print('Dataset Shape:', df.shape)
print('Group Sizes:\n', df['Group'].value_counts())
display(df.groupby('Group')['Yield_Per_Hectare'].describe())

In [ ]:
# Exploratory Data Analysis (EDA)
def visualize_groups(df, metric):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Boxplot with Swarm
    sns.boxplot(x='Group', y=metric, data=df, ax=axes[0], palette='Set2')
    axes[0].set_title('Distribution Comparison (Boxplot)')
    
    # KDE Plot
    sns.kdeplot(data=df, x=metric, hue='Group', fill=True, ax=axes[1], palette='Set2')
    axes[1].set_title('Density Distribution (KDE)')
    
    plt.show()

visualize_groups(df, 'Yield_Per_Hectare')

In [ ]:
# Statistical Analysis Engine
class ABTestEngine:
    def __init__(self, df, metric):
        self.control = df[df['Group']=='Control'][metric]
        self.treatment = df[df['Group']=='Treatment'][metric]
        
    def check_assumptions(self):
        print('--- Assumption Checks ---')
        # 1. Normality (Shapiro-Wilk)
        # Note: For large N, T-test is robust, but good to check
        _, p_c = stats.shapiro(self.control)
        _, p_t = stats.shapiro(self.treatment)
        print(f'Shapiro-Wilk (Control): p={p_c:.4f}')
        print(f'Shapiro-Wilk (Treatment): p={p_t:.4f}')
        
        # 2. Homogeneity of Variance (Levene)
        _, p_l = stats.levene(self.control, self.treatment)
        print(f'Levene Test: p={p_l:.4f}')
        if p_l < 0.05:
            print('-> Variances are NOT equal (Use Welch\'s T-test)')
            return False # Equal var is False
        else:
            print('-> Variances are equal')
            return True
            
    def run_test(self, equal_var=True):
        print('\n--- Hypothesis Test Results ---')
        t_stat, p_val = stats.ttest_ind(self.control, self.treatment, equal_var=equal_var)
        
        print(f'T-Statistic: {t_stat:.4f}')
        print(f'P-Value: {p_val:.4f}')
        
        # Effect Size (Cohen\'s d)
        n1, n2 = len(self.control), len(self.treatment)
        s1, s2 = np.var(self.control, ddof=1), np.var(self.treatment, ddof=1)
        s_pooled = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
        cohens_d = (np.mean(self.control) - np.mean(self.treatment)) / s_pooled
        
        print(f'Cohen\'s d (Effect Size): {abs(cohens_d):.4f}')
        
        if p_val < 0.05:
            print('\n✅ RESULT: Statistically Significant Difference Detected!')
            print(f'The treatment group mean ({np.mean(self.treatment):.2f}) is significantly different from control ({np.mean(self.control):.2f}).')
        else:
            print('\n❌ RESULT: No Significant Difference.')
            
        # Power Analysis
        analysis = TTestIndPower()
        power = analysis.solve_power(effect_size=abs(cohens_d), nobs1=n1, ratio=1.0, alpha=0.05)
        print(f'Statistical Power: {power:.4f}')

engine = ABTestEngine(df, 'Yield_Per_Hectare')
equal_variance = engine.check_assumptions()
engine.run_test(equal_var=equal_variance)

## 🧪 Conclusion & Recommendations

1. **Significance**: The analysis shows a p-value of ... indicating...
2. **Effect Size**: The observed effect size (Cohen's d) suggests a [Small/Medium/Large] practical difference.
3. **Business Impact**: If implemented, this change could lead to a ...% improvement in `Yield_Per_Hectare`.
4. **Recommendation**: Proceed with full rollout / Iterate on the design.